## Semantic-Based Topic Modeling

In [ ]:
from datetime import datetime
date = datetime.now()
formatted_date = date.strftime("%B %d, %Y")
print(formatted_date)

April 07, 2025


In [ ]:
# Check GPU memory
!nvidia-smi

Mon Apr  7 02:37:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Check system RAM
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       1.5Gi        57Gi       6.0Mi        24Gi        81Gi
Swap:             0B          0B          0B


#### Setting up the computing environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

# Set up the current working directory within the Google Drive
%cd /content/drive/My\ Drive/Colab\ Notebooks/LLM/sped_biblio/topic_modeling

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/Colab Notebooks/LLM/sped_biblio/topic_modeling


In [ ]:
!pip install --upgrade -q pandas==2.2.2 numpy==1.26.4 sentence-transformers bertopic scikit-learn matplotlib umap-learn hdbscan datamapplot
!pip install -q dill qgrid
# !pip install -U sentence-transformers transformers
# !pip install spacy
# !python -m spacy download en_core_web_md
# !pip install --upgrade bertopic
!pip install -q python-dotenv
!pip install -q openai --upgrade

In [ ]:
#!pip uninstall -y pandas pandas
#!pip install --upgrade pandas==2.2.2 numpy==1.26.4

In [ ]:
import re
import warnings
from collections import defaultdict
import pickle
from pickle import UnpicklingError
from pickle import PicklingError
import itertools

# Data Manipulation
import dill
import numpy as np
import pandas as pd
import requests
import qgrid
import requests
import os
import re

# Natural Language Processing
import nltk
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

import spacy

# Generating Topic Labels
import openai
from dotenv import load_dotenv

# Clustering
from hdbscan import HDBSCAN
from umap import UMAP
from scipy.cluster import hierarchy as sch

# Visualization Imports
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.ticker import FuncFormatter
import colorlover as cl
import textwrap
from scipy.cluster import hierarchy as sch

# Network Analysis
import networkx as nx

# Progress Bar
from tqdm import tqdm

# Display HTML
from IPython.display import IFrame

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

def generate_topic_labels(topic_info, topic_model, openai_model="gpt-4o", max_tokens=10, temperature=0.3):

    gen_names = []

    for topic_id in topic_info['Topic']:
        if topic_id == -1:
            gen_names.append("Outlier")
            continue

        keywords = topic_model.get_topic(topic_id)
        if keywords:
            top_keywords = ", ".join([keyword[0] for keyword in keywords[:10]])
            prompt = f"""
You are a highly skilled data scientist specializing in generating concise and descriptive topic labels based on provided top terms for each topic.
Each topic consists of a list of terms ordered from most to least significant.

Your objective is to create precise and concise labels that capture the essence of each topic by following these guidelines:

1. Use Person-First Language:
   - Prioritize respectful and inclusive language.
   - Avoid terms that may be considered offensive or stigmatizing.
   - For example, use "students with learning disabilities" instead of "disabled students".

2. Analyze the significance of the top terms:
   - Focus primarily on the most significant terms.
   - Include additional terms if they add essential context.

3. Synthesize the Topic Label:
   - Ensure clarity and conciseness (aim for 5 words).
   - Reflect the collective meaning of the most influential terms.
   - Use descriptive yet precise phrasing.

4. Maintain consistency:
   - Capitalize the first word using title case.
   - Use uniform formatting and avoid ambiguity.
   - Make consie and complete expressions.

Example
----------
Top 10 Keywords in [Representation]:
virtual manipulatives, manipulatives, mathematical, app, solving, learning disability, algebra, area, tool, concrete manipulatives

Generated Topic Label in [GenName]:
Visual-based technology for mathematical problem solving

Top 10 Keywords: {top_keywords}
Generated Topic Label in [GenName]:
"""
            response = openai.chat.completions.create(
                model=openai_model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=temperature
            )
            gen_name = response.choices[0].message.content.strip()
            gen_name = re.sub(r'Generated Topic Label in \[GenName\]:', '', gen_name, flags=re.IGNORECASE).strip()
            gen_names.append(gen_name)
        else:
            gen_names.append("No Keywords")

    topic_info['GenName'] = gen_names
    return topic_info

#### Combine text columns

In [ ]:
all_data_file = f"files/all_data.xlsx"
all_data = pd.read_excel(all_data_file, na_filter=False)

df = all_data[all_data['filtered'] == 'Yes'].reset_index(drop=True)
df['Year'] = df['PY'].astype(int)
df['Decade'] = (df['Year'] // 10) * 10
df['CR'] = df['CR'].astype(str)

#### Preprocess documents

In [ ]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
publication_embeddings = sentence_model.encode(df['combined_text'].tolist(), show_progress_bar=True)
publication_embeddings_df = pd.DataFrame(publication_embeddings)
publication_embeddings_df['UT'] = df['UT'].tolist()

Batches:   0%|          | 0/108 [00:00<?, ?it/s]

#### Conduct topic modeling

In [ ]:
umap_model = UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.00,
    metric='cosine',
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(publication_embeddings_df.drop(columns='UT'))
reduced_embeddings_df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
reduced_embeddings_df['UT'] = publication_embeddings_df['UT'].tolist()

hdbscan_model = HDBSCAN(
    min_cluster_size=35,
    min_samples=40,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

hdbscan_model.fit(reduced_embeddings)
labels = hdbscan_model.labels_

vectorizer_model =  CountVectorizer(min_df=10)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,
  calculate_probabilities=True,
  verbose=True
)

topics, probs = topic_model.fit_transform(df['combined_text'])

2025-04-07 02:47:27,734 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2025-04-07 02:47:30,563 - BERTopic - Embedding - Completed ✓
2025-04-07 02:47:30,564 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-07 02:47:47,297 - BERTopic - Dimensionality - Completed ✓
2025-04-07 02:47:47,298 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-07 02:47:47,533 - BERTopic - Cluster - Completed ✓
2025-04-07 02:47:47,537 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-07 02:47:48,602 - BERTopic - Representation - Completed ✓


In [ ]:
df["combined_text"] = (
    df["combined_text"]
    .str.replace(
        r"(?i)\baugmentative and alternative communication\b",
        "AAC",
        regex=True
    )
    .str.replace(
        r"(?i)\bapplied behavior analysis\b",
        "ABA",
        regex=True
    )
    .str.replace(
        r"(?i)\bautism spectrum disorder\b",
        "ASD",
        regex=True
    )
)

vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 3), min_df=10)
topic_model.update_topics(df['combined_text'], vectorizer_model=vectorizer_model)

In [ ]:
# topic_model.get_topic_info()

In [ ]:
new_topics = topic_model.reduce_outliers(df['combined_text'], topics, strategy="c-tf-idf")

In [ ]:
topic_model.update_topics(df['combined_text'], topics=new_topics, vectorizer_model=vectorizer_model)

In [ ]:
# topic_model.get_topic_info()

In [ ]:
df['Topic'] = [topic + 1 for topic in new_topics]

topic_info = topic_model.get_topic_info()

topic_info_gen = generate_topic_labels(topic_info, topic_model)

topic_model.set_topic_labels(topic_info_gen['GenName'].tolist())

for i in range(probs.shape[1]):
    df[f'Prob_Topic_{i}'] = probs[:, i]

data = []
for topic in topic_info['Topic']:
    words_with_scores = topic_model.get_topic(topic)
    for word, score in words_with_scores:
        data.append({"Topic": topic, "Word": word, "Score": score})

topic_model_words = pd.DataFrame(data)

topic_info_words = topic_info.merge(topic_model_words, how="left", on="Topic") \
                                                      .sort_values(by=['Topic', 'Score'], ascending=[True, False])

topic_info['Topic'] = topic_info['Topic'] + 1
topic_info_words['Topic'] = topic_info_words['Topic'] + 1

In [ ]:
topic_info.to_excel("files/topic_info.xlsx", engine='openpyxl', index=False)
topic_info_words.to_excel("files/topic_info_words.xlsx", engine='openpyxl', index=False)
df.to_excel("files/df.xlsx", engine='openpyxl', index=False)

In [ ]:
# topic_info = pd.read_excel("files/topic_info.xlsx")
# topic_info_words = pd.read_excel("files/topic_info_words.xlsx")
# df = pd.read_excel("files/df.xlsx")
topic_info_human_loop = pd.read_excel("files/topic_info_human_loop.xlsx")

In [ ]:
# topic_info = topic_info.drop(columns=['Topic'])
# topic_info_concat = pd.merge(topic_info, topic_info_human_loop[['Name', 'Topic', 'CustomLabel']], how="left", on="Name")
topic_info_concat = pd.merge(topic_info, topic_info_human_loop[['Name', 'CustomLabel']], how="left", on="Name")
order = ['Topic', 'Count', 'Name', 'Representation', 'GenName', 'CustomLabel']
topic_info_concat = topic_info_concat.reindex(columns=order)

# topic_info_words = topic_info_words.drop(columns=['Topic'])
# topic_info_words_concat = pd.merge(topic_info_words, topic_info_human_loop[['Name', 'Topic', 'CustomLabel']], how="left", on="Name")
topic_info_words_concat = pd.merge(topic_info_words, topic_info_human_loop[['Name', 'CustomLabel']], how="left", on="Name")
order = ['Topic', 'Count', 'Name', 'Representation', 'Word', 'Score', 'Representative_Docs', 'GenName', 'CustomLabel']
topic_info_words_concat = topic_info_words_concat.reindex(columns=order)

In [ ]:
files = {
    "files/topic_model.pkl": topic_model,
    "files/publication_embeddings.pkl": publication_embeddings,
    "files/publication_embeddings_df.pkl": publication_embeddings_df,
    "files/reduced_embeddings.pkl": reduced_embeddings,
    "files/reduced_embeddings_df.pkl": reduced_embeddings_df,
    "files/topic_info.pkl": topic_info,
    "files/topic_info_concat.pkl": topic_info_concat,
    "files/topic_info_words.pkl": topic_info_words,
    "files/topic_info_words_concat.pkl": topic_info_words_concat,
    "files/df.pkl": df
}

for filename, data in files.items():
    if data is None:
        continue

    try:
        with open(filename, "wb") as f:
            pickle.dump(data, f)
    except Exception as e:
        print(f"Error saving pickle for {filename}: {e}")

    try:
        excel_filename = filename.rsplit('.', 1)[0] + ".xlsx"
        if isinstance(data, pd.DataFrame):
            data.to_excel(excel_filename, index=False)
        elif isinstance(data, np.ndarray) and data.ndim in [1, 2]:
            pd.DataFrame(data).to_excel(excel_filename, index=False)
    except Exception as e:
        print(f"Error saving Excel for {filename}: {e}")

In [ ]:
#def load_data_from_files(files_dict):
#   loaded_data = {}
#   for filename, var_name in files_dict.items():
#       filepath = os.path.join('files', filename)
#       if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
#           try:
#               with open(filepath, "rb") as f:
#                   loaded_data[var_name] = pickle.load(f)
#           except Exception as e:
#               loaded_data[var_name] = None
#               print(f"Error loading {filepath}: {e}")
#       else:
#           loaded_data[var_name] = None
#   return loaded_data

#files_to_load = {
#   "topic_model.pkl": "topic_model",
#   "publication_embeddings.pkl": "publication_embeddings",
#   "publication_embeddings_df.pkl": "publication_embeddings_df",
#   "reduced_embeddings.pkl": "reduced_embeddings",
#   "reduced_embeddings_df.pkl": "reduced_embeddings_df",
#   "topic_info.pkl": "topic_info",
#   "topic_info_concat.pkl": "topic_info_concat",
#   "topic_info_words.pkl": "topic_info_words",
#   "topic_info_words_concat.pkl": "topic_info_words_concat",
#   "df.pkl": "df"
#}

#loaded_data = load_data_from_files(files_to_load)
#topic_model = loaded_data.get("topic_model")
#publication_embeddings = loaded_data.get("publication_embeddings")
#publication_embeddings_df = loaded_data.get("publication_embeddings_df")
#reduced_embeddings = loaded_data.get("reduced_embeddings")
#reduced_embeddings_df = loaded_data.get("reduced_embeddings_df")
#topic_info = loaded_data.get("topic_info")
#topic_info_concat = loaded_data.get("topic_info_concat")
#topic_info_words = loaded_data.get("topic_info_words")
#topic_info_words_concat = loaded_data.get("topic_info_words_concat")
#df = loaded_data.get("df")

#### Identify representative words and scores for each topic

In [ ]:
topic_word_data = topic_info_words_concat.copy()

topic_word_data['Word (c-TF-IDF)'] = topic_word_data.apply(
    lambda row: f"{row['Word']} ({row['Score']:.3f})", axis=1
)

topic_word_data = topic_word_data.sort_values(by=['CustomLabel', 'Score'], ascending=[True, False])

topic_word_table = topic_word_data.pivot_table(
    index='CustomLabel',
    values=['Word (c-TF-IDF)', 'Topic'],
    aggfunc=lambda x: ", ".join(x.astype(str)) if x.dtype == 'object' else x.iloc[0]
).reset_index()

topic_counts = topic_word_data.groupby('CustomLabel')['Count'].first().reset_index()

topic_word_table = pd.merge(topic_word_table, topic_counts, on='CustomLabel', how='left')

topic_word_table = topic_word_table[['Topic', 'CustomLabel', 'Count', 'Word (c-TF-IDF)']]

topic_word_table = topic_word_table.sort_values(by='Topic')

styles = [
    dict(selector="caption", props=[("caption-side", "top")]),
    {'selector': 'th', 'props': [('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'left')]}
]

topic_word_table = (
    topic_word_table.style
    .set_caption("<b>Topic Word Score</b>")
    .set_table_styles(styles)
    .set_properties(**{
        'font-size': '24px',
        'font-family': 'Helvetica Neue',
        'color': 'black',
        'text-align': 'center'
    })
    .set_properties(**{
        'text-align': 'center',
        'font-size': '14px',
        'font-family': 'Helvetica Neue',
        'color': 'black',
    }, subset=pd.IndexSlice[:, topic_word_table.columns])
    .hide(axis='index')
)

In [ ]:
topic_word_table

Topic,CustomLabel,Count,Word (c-TF-IDF)
1,Video modeling,571,"video (0.054), skills (0.045), social (0.041), modeling (0.036), asd (0.034), children (0.033), video modeling (0.030), autism (0.027), teaching (0.022), prompting (0.022)"
2,Function-based interventions,336,"reinforcement (0.078), behavior (0.061), response (0.037), treatment (0.033), functional (0.032), automatic (0.031), assessment (0.029), functional analysis (0.029), analysis (0.029), problem behavior (0.028)"
3,Rehabilitation interventions,283,"design (0.027), study (0.027), single (0.026), rehabilitation (0.024), brain (0.023), injury (0.023), intervention (0.022), subject (0.022), memory (0.021), subjects (0.019)"
4,Communication and speech,278,"aac (0.066), communication (0.065), speech (0.042), children (0.039), intervention (0.029), asd (0.029), autism (0.026), language (0.026), use (0.022), using (0.019)"
5,Mental health treatments,270,"treatment (0.032), health (0.030), anxiety (0.028), study (0.027), intervention (0.027), therapy (0.025), self (0.024), participants (0.024), design (0.024), case (0.023)"
6,Reading instruction,206,"reading (0.102), students (0.051), words (0.041), word (0.034), instruction (0.032), writing (0.031), intervention (0.028), learning (0.025), fluency (0.025), disabilities (0.024)"
7,Single-case data and analysis,193,"case (0.059), single (0.058), data (0.057), single case (0.056), analysis (0.042), research (0.033), designs (0.032), design (0.031), visual (0.030), effect (0.025)"
8,Staff training,214,"training (0.081), staff (0.039), instruction (0.037), participants (0.028), skills (0.025), feedback (0.025), teaching (0.024), stimulus (0.023), relations (0.023), based (0.022)"
9,Mathematics instruction,196,"students (0.086), learning (0.051), virtual (0.033), instruction (0.032), design (0.027), study (0.027), disabilities (0.026), solving (0.026), education (0.025), research (0.024)"
10,Telehealth training,126,"telehealth (0.106), services (0.044), covid (0.042), covid 19 (0.041), 19 (0.040), training (0.035), pandemic (0.035), aba (0.034), parent (0.032), behavior (0.030)"


#### Visualize the semantic embedding space of documents by topic

In [66]:
color_map = [
    '#636EFA',
    '#EF553B',
    '#00CC96',
    '#AB63FA',
    '#FFA15A',
    '#19D3F3',
    '#FF6692',
    '#B6E880',
    '#FF97FF',
    '#FECB52',
    '#A9A9A9',
    '#FF7F50',
    '#8B4513',
    '#00CED1',
    '#556B2F',
    '#DC143C',
]

docs_topics_data = pd.merge(df, topic_info_concat[['Topic', 'CustomLabel']], on='Topic', how='left')
docs_topics_data['Topic'] = pd.to_numeric(docs_topics_data['Topic'], errors='coerce')
docs_topics_data = pd.merge(docs_topics_data, reduced_embeddings_df[['UT', 'x', 'y']], on='UT', how='left')
docs_topics_data['x'] = pd.to_numeric(docs_topics_data['x'], errors='coerce')
docs_topics_data['y'] = pd.to_numeric(docs_topics_data['y'], errors='coerce')
docs_topics_data = docs_topics_data.dropna(subset=['Topic', 'x', 'y'])
docs_topics_data['Topic'] = docs_topics_data['Topic'].astype(int)

ordered_legend = docs_topics_data.sort_values('Topic')['CustomLabel'].unique().tolist()
docs_topics_data['legend_topic'] = pd.Categorical(docs_topics_data['CustomLabel'],
                                                    categories=ordered_legend,
                                                    ordered=True)

In [ ]:
docs_topics_data['hover_text'] = (
    "Title: " +
    docs_topics_data['TI'].apply(lambda text: "<br>".join(textwrap.wrap(text, width=50))) +
    "<br><br>Topic " + docs_topics_data['Topic'].astype(str) + ": " + docs_topics_data['CustomLabel']
)

color_mapping = {cat: color_map[i % len(color_map)] for i, cat in enumerate(ordered_legend)}

fig_cluster = px.scatter(
    docs_topics_data,
    x='x',
    y='y',
    color='legend_topic',
    color_discrete_map=color_mapping,
    labels={'x': 'X', 'y': 'Y'},
    custom_data=['hover_text'],
    category_orders={'legend_topic': ordered_legend}
)

fig_cluster.update_traces(
    marker=dict(size=9, opacity=0.7, line=dict(width=1, color="white")),
    selector=dict(mode='markers'),
    hovertemplate='<b>%{customdata[0]}</b><extra></extra>'
)

fig_cluster.update_layout(
    title="<b>Semantic Space of Documents by Topic</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=80, b=80, l=80, r=80),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=800,
    height=800,
    showlegend=True,
    legend_title_text="Topic"
)

fig_cluster.write_html("results/docs_topics.html")
#fig_cluster.show()

In [68]:
IFrame(src='results/docs_topics.html', width=800, height=800)

#### Calculate cosine similarity among topics

In [ ]:
merged_df = pd.merge(df, publication_embeddings_df, on='UT')

embeddings = [col for col in publication_embeddings_df.columns if col != 'UT']
topic_embeddings = merged_df.groupby("Topic")[embeddings].mean()

cosine_sim_matrix = cosine_similarity(topic_embeddings)

cosine_sim_df = pd.DataFrame(
    cosine_sim_matrix,
    index=topic_embeddings.index,
    columns=topic_embeddings.index
)

fig_cosine_sim = px.imshow(
    cosine_sim_df,
    color_continuous_scale="RdBu_r",
    origin="lower",
    labels=dict(color="Cosine Similarity"),
    x=cosine_sim_df.columns,
    y=cosine_sim_df.index
)

fig_cosine_sim.update_xaxes(
    tickmode='array',
    tickvals=cosine_sim_df.columns,
    ticktext=[str(topic) for topic in cosine_sim_df.columns]
)
fig_cosine_sim.update_yaxes(
    tickmode='array',
    tickvals=cosine_sim_df.index,
    ticktext=[str(topic) for topic in cosine_sim_df.index]
)

fig_cosine_sim.update_layout(
    title="<b>Cosine Similarity Among Topics</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=80, b=80, l=80, r=80),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=800,
    height=800,
    showlegend=True,
    legend_title_text="Topic"
)

fig_cosine_sim.write_html("results/fig_cosine_sim.html")
# fig_cosine_sim.show()

In [72]:
IFrame(src='results/fig_cosine_sim.html', width=800, height=800)

#### Ngrams

In [73]:
vectorizer = vectorizer_model.build_analyzer()

def extract_ngrams(text):
    all_ngrams = vectorizer(text)
    unigrams = [ng for ng in all_ngrams if len(ng.split()) == 1]
    bigrams  = [ng for ng in all_ngrams if len(ng.split()) == 2]
    trigrams = [ng for ng in all_ngrams if len(ng.split()) == 3]
    return unigrams, bigrams, trigrams

df['unigrams'], df['bigrams'], df['trigrams'] = zip(*df['combined_text'].apply(extract_ngrams))

In [74]:
df = docs_topics_data.copy()
with open("files/df.pkl", "wb") as f_df:
    pickle.dump(df, f_df)

df.to_excel("files/df.xlsx", engine='openpyxl', index=False)
topic_word_table.to_excel("files/topic_word_table.xlsx", engine='openpyxl', index=False)
cosine_sim_df.to_excel("files/cosine_sim_df.xlsx", engine='openpyxl', index=False)

In [ ]:
# with open("files/df.pkl", "rb") as f_df:
#     df = pickle.load(f_df)

# df = pd.read_excel("files/df.xlsx")
# topic_word_table = pd.read_excel("files/topic_word_table.xlsx")
# cosine_sim_df = pd.read_excel("files/cosine_sim_df.xlsx")

In [79]:
from nbconvert import HTMLExporter
import nbformat

notebook_path = 'index.ipynb'
html_exporter = HTMLExporter()

with open(notebook_path, 'r', encoding='utf-8') as nb_file:
    notebook_content = nb_file.read()
    notebook = nbformat.reads(notebook_content, as_version=4)

if 'widgets' in notebook.metadata and 'application/vnd.jupyter.widget-state+json' in notebook.metadata['widgets']:
    if 'state' not in notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']:
        notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']['state'] = {}

html_output, _ = html_exporter.from_notebook_node(notebook)

with open('index.html', 'w', encoding='utf-8') as html_file:
    html_file.write(html_output)